# Lab: The Mean Value Theorem, Linear Approximation, and Euler's Method

Math 140H, Honors Calculus I

In the Week 9 lab we built $\exp$ with Euler's method and saw on a log-log plot that the error shrinks roughly in proportion to the step size $h$. We *observed* that; we did not *explain* it. This lab supplies the explanation. The main tool is the Mean Value Theorem, used several times in a row:

1. **Part 1.** The MVT, and the MVT as an *estimate* for $f(b)-f(a)$.
2. **Part 2.** The tangent line as a linear approximation, and how fast its error goes to $0$.
3. **Part 3.** The MVT applied to $f$, *and then to $f'$*, gives a rigorous error bound for linear approximation. (New material.)
4. **Part 4.** Euler's method is linear approximation, repeated. Part 3 plus some bookkeeping gives a rigorous error bound for Euler's method on a harder example than $y'=y$.
5. **Extensions.** Euler's method as a Riemann sum in disguise, and a falling object with air resistance.

### How this notebook works

Every computation in this notebook is meant to **confirm something you have already worked out by hand**.

- Before each checking code cell there is a box labeled **✏️ P1.1, P1.2, …**. Do that task **on paper first**.
- Then type your paper answer into the line marked `# <- P...` in the code cell, and run the cell. The cell compares your answer with the computer's. If you leave an answer as `None`, the cell will **not** reveal the computer's value.
- Enter exact expressions where you have them, in Python syntax: `2/np.sqrt(3)`, `np.log(np.e - 1)`, `1 + np.pi/4`. Decimals rounded to 4 significant digits also pass.
- If your answer disagrees, **do not erase** your original work. Find the mistake and write the correction next to it, with one sentence on what went wrong.
- Tasks marked *(reflect)* need a few sentences, not a number.
- A calculator is fine for plain arithmetic (evaluating $e^{\pi}$, say). All derivations, exact values and estimates must be on paper.

**What to submit:** your handwritten pages, with every P-task labeled and in order, as a supplement to this notebook.

In [1]:
# --- Setup: run this cell first. You do not need to change anything here. ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True


def _missing(label):
    print(f"[{label}] No paper answer entered yet. Do {label} on paper, type your answer in, and re-run.")


def check(label, yours, computed, rel_tol=1e-3):
    # Compare one number you worked out on paper with the computer's value.
    if yours is None:
        _missing(label)
        return False
    yours, computed = float(yours), float(computed)
    ok = abs(yours - computed) <= rel_tol * max(abs(computed), 1e-12) + 1e-12
    status = "✓ agrees" if ok else "✗ does NOT agree: recheck your paper work"
    print(f"[{label}] paper: {yours:.10g}   computer: {computed:.10g}   {status}")
    return ok


def check_list(label, yours, computed, rel_tol=1e-3, ordered=False):
    # Same as check, for a list of numbers. If ordered=False the order does not matter.
    if yours is None:
        _missing(label)
        return False
    yours, computed = [float(v) for v in yours], [float(v) for v in computed]
    if not ordered:
        yours, computed = sorted(yours), sorted(computed)
    if len(yours) != len(computed):
        print(f"[{label}] ✗ you listed {len(yours)} value(s), the computer found {len(computed)}: recheck.")
        return False
    if len(computed) == 0:
        print(f"[{label}] ✓ agrees: there are no such values.")
        return True
    results = [check(f"{label} #{i+1}", y, c, rel_tol) for i, (y, c) in enumerate(zip(yours, computed))]
    return all(results)


def check_bound(label, your_bound, actual):
    # Check that a bound you derived on paper really is an upper bound for |actual|.
    if your_bound is None:
        _missing(label)
        return False
    actual = abs(float(actual))
    your_bound = float(your_bound)
    if actual <= your_bound:
        print(f"[{label}] ✓ valid bound:  |actual| = {actual:.6g}  <=  your bound = {your_bound:.6g}"
              f"   (actual/bound = {actual / your_bound:.3g}; 1 would be perfectly sharp)")
        return True
    print(f"[{label}] ✗ NOT a valid bound: |actual| = {actual:.6g} is bigger than your bound {your_bound:.6g}")
    return False


def compare_prediction(label, guess, actual):
    # For guesses: no pass/fail, just how close you were.
    if guess is None:
        _missing(label)
        return
    guess, actual = float(guess), float(actual)
    print(f"[{label}] your prediction: {guess:.6g}   actual: {actual:.6g}   "
          f"(actual/prediction = {actual / guess:.3g})" if guess != 0 else
          f"[{label}] your prediction: 0   actual: {actual:.6g}")


def answered(*vals):
    # True if all paper answers have been entered.
    return all(v is not None for v in vals)

## Part 1: The Mean Value Theorem

> **Mean Value Theorem.** Let $f$ be continuous on $[a,b]$ and differentiable on $(a,b)$. Then there is a $c\in(a,b)$ with
> $$f'(c) = \frac{f(b)-f(a)}{b-a}.$$

Geometrically: somewhere strictly between $a$ and $b$, the tangent line is parallel to the secant line through $(a,f(a))$ and $(b,f(b))$.

Compare the Mean Value Theorem for Integrals, $\int_a^b f(x)\,dx = f(c)(b-a)$. Both theorems assert that *some* $c$ exists and say nothing about where it is. In this part we find $c$ anyway: first on paper, then by computer.

### Example 1: a cubic

We use $f(x) = x^3-3x$, the same function as in the Week 2 continuity widget.

> **✏️ P1.1** On the interval $[0,2]$:
> (a) Compute the secant slope $m = \dfrac{f(2)-f(0)}{2-0}$.
> (b) Solve $f'(c) = m$ exactly. Which solutions lie in $(0,2)$?

In [2]:
# Helper functions for Part 1 (no need to change anything in the first half of this cell)

def mvt_points(df, a, b, m, n_grid=20001):
    # All c in (a, b) with df(c) = m: look for sign changes of df - m on a fine grid,
    # then refine each one with brentq. (A tangency where df - m touches 0 without
    # changing sign would be missed; none of our examples have one.)
    xs = np.linspace(a, b, n_grid)[1:-1]
    g = df(xs) - m
    cs = []
    for i in range(len(xs) - 1):
        if g[i] == 0:
            cs.append(xs[i])
        elif g[i] * g[i + 1] < 0:
            c = brentq(lambda x: df(x) - m, xs[i], xs[i + 1])
            if abs(df(c) - m) < 1e-8:      # discard jumps of df that only *look* like roots
                cs.append(c)
    return cs


def plot_mvt(f, df, a, b, cs, title):
    pad = 0.15 * (b - a)
    xs = np.linspace(a - pad, b + pad, 400)
    m = (f(b) - f(a)) / (b - a)
    plt.plot(xs, f(xs), label='f')
    plt.plot([a, b], [f(a), f(b)], 'ko-', label=f'secant, slope {m:.4g}')
    for c in cs:
        t = np.linspace(c - 0.3 * (b - a), c + 0.3 * (b - a), 2)
        plt.plot(t, f(c) + df(c) * (t - c), 'r--')
        plt.plot(c, f(c), 'ro')
    if cs:
        plt.plot([], [], 'r--', label='tangent at c')
    plt.axvline(a, color='gray', lw=0.6); plt.axvline(b, color='gray', lw=0.6)
    plt.title(title); plt.legend(); plt.show()


f1  = lambda x: x**3 - 3*x
df1 = lambda x: 3*x**2 - 3

# ------------------------------------------------------------------ P1.1
m_paper = None    # <- P1.1(a): the secant slope
c_paper = None    # <- P1.1(b): list of all c in (0, 2), e.g. [0.5] or [0.3, 1.2]

a, b = 0, 2
m = (f1(b) - f1(a)) / (b - a)
cs = mvt_points(df1, a, b, m)
check('P1.1(a)', m_paper, m)
check_list('P1.1(b)', c_paper, cs)
if answered(c_paper):
    plot_mvt(f1, df1, a, b, cs, 'MVT for x^3 - 3x on [0, 2]')

[P1.1(a)] No paper answer entered yet. Do P1.1(a) on paper, type your answer in, and re-run.
[P1.1(b)] No paper answer entered yet. Do P1.1(b) on paper, type your answer in, and re-run.


> **✏️ P1.2** Repeat P1.1 for the same function on $[-2,2]$. How many values of $c$ are there this time?
> *(reflect)* Does this contradict anything in the statement of the MVT? What exactly does the theorem promise?

In [ ]:
m_paper = None    # <- P1.2: the secant slope on [-2, 2]
c_paper = None    # <- P1.2: list of all c in (-2, 2)

a, b = -2, 2
m = (f1(b) - f1(a)) / (b - a)
cs = mvt_points(df1, a, b, m)
check('P1.2 slope', m_paper, m)
check_list('P1.2 c', c_paper, cs)
if answered(c_paper):
    plot_mvt(f1, df1, a, b, cs, 'MVT for x^3 - 3x on [-2, 2]')

### Example 2: the exponential function

> **✏️ P1.3** Let $f(x)=e^x$ on $[0,1]$.
> (a) Find $c$ exactly (your answer will involve $\ln$), then as a decimal.
> (b) *(reflect)* Explain, using only the fact that $f'$ is strictly increasing, why there can be **only one** such $c$.

In [ ]:
c_paper = None    # <- P1.3(a): the value of c (a single number this time)

a, b = 0, 1
m = (np.exp(b) - np.exp(a)) / (b - a)
cs = mvt_points(np.exp, a, b, m)
check_list('P1.3', None if c_paper is None else [c_paper], cs)
if answered(c_paper):
    plot_mvt(np.exp, np.exp, a, b, cs, 'MVT for e^x on [0, 1]')

### Example 3: when a hypothesis fails

> **✏️ P1.4** Let $f(x)=|x|$ on $[-1,2]$.
> (a) Compute the secant slope $m$.
> (b) Show that **no** $c\in(-1,2)$ satisfies $f'(c)=m$. Which hypothesis of the MVT fails, and where?
>
> In the code cell, enter an empty list `[]` if you conclude there is no such $c$.

In [ ]:
c_paper = None    # <- P1.4(b): list of all c in (-1, 2) with f'(c) = m

f3  = np.abs
df3 = np.sign      # the derivative of |x| for x != 0 (at x = 0 there is no derivative)

a, b = -1, 2
m = (f3(b) - f3(a)) / (b - a)
cs = mvt_points(df3, a, b, m)
check_list('P1.4', c_paper, cs)
if answered(c_paper):
    plot_mvt(f3, df3, a, b, cs, '|x| on [-1, 2]: no tangent parallel to the secant')

### The MVT as an estimate

Usually we cannot solve $f'(c)=m$, and we do not need to. If we know bounds $m_1 \le f'(x) \le M_1$ for all $x\in(a,b)$, then the MVT immediately gives
$$
m_1\,(b-a) \;\le\; f(b)-f(a) \;\le\; M_1\,(b-a).
$$
This has exactly the same shape as the min–max bound for integrals, $m(b-a)\le\int_a^b f \le M(b-a)$.

> **✏️ P1.5** Apply this to $f(x)=\sqrt{x}$ on $[100,101]$.
> (a) Show that $\sqrt{101} < 10 + \frac{1}{20} = 10.05$.
> (b) Show that $\sqrt{101} > 10 + \dfrac{1}{2\sqrt{101}}$. This lower bound still contains $\sqrt{101}$. Use (a) to replace it by a bound involving only plain numbers.
> (c) Write your lower and upper bounds as decimals (8 significant digits), and the width of the interval between them.

In [ ]:
lower_paper = None   # <- P1.5(b): your lower bound for sqrt(101)
upper_paper = None   # <- P1.5(a): your upper bound for sqrt(101)

if answered(lower_paper, upper_paper):
    true = np.sqrt(101)
    print(f"your interval:  {lower_paper:.10f} < sqrt(101) < {upper_paper:.10f}")
    print(f"sqrt(101) =     {true:.10f}")
    if lower_paper < true < upper_paper:
        print(f"✓ both bounds are valid. Width of your interval: {upper_paper - lower_paper:.3g}")
    else:
        print("✗ at least one bound is wrong: recheck your work.")
else:
    _missing('P1.5')

## Part 2: Linear approximation

The tangent line to $f$ at $a$ is $L(x) = f(a) + f'(a)(x-a)$. Writing $x = a+h$, we split $f(a+h)$ into the tangent-line part and an error:
$$
f(a+h) = \underbrace{f(a) + f'(a)\,h}_{\text{linear approximation}} + E(h),
\qquad
E(h) := f(a+h) - f(a) - f'(a)\,h .
$$
Notice that the upper bound $10.05$ in P1.5 is exactly $L(101)$ for $f(x)=\sqrt x$ and $a=100$. That is not a coincidence; you will see why in P3.3.

> **✏️ P2.1** For each case, write down $L$, compute the approximation (exact, then decimal), and **predict** whether it is too big ("over") or too small ("under"). Base your prediction on a sketch of the graph and its tangent line, not on a calculator.
> (a) $\sqrt{101}$, using $f(x)=\sqrt x$, $a=100$, $h=1$.
> (b) $e^{0.1}$, using $f(x)=e^x$, $a=0$, $h=0.1$.
> (c) $\ln(0.9)$, using $f(x)=\ln x$, $a=1$, $h=-0.1$.

In [ ]:
approx_a_paper = None   # <- P2.1(a): linear approximation of sqrt(101)
approx_b_paper = None   # <- P2.1(b): linear approximation of e^0.1
approx_c_paper = None   # <- P2.1(c): linear approximation of ln(0.9)

predict_a = None        # <- 'over' or 'under'
predict_b = None
predict_c = None

cases = {   # name: (f, f', a, h)
    'a': (np.sqrt, lambda x: 0.5 / np.sqrt(x), 100.0, 1.0),
    'b': (np.exp,  np.exp,                     0.0,   0.1),
    'c': (np.log,  lambda x: 1 / x,            1.0,  -0.1),
}
yours    = {'a': approx_a_paper, 'b': approx_b_paper, 'c': approx_c_paper}
predicts = {'a': predict_a, 'b': predict_b, 'c': predict_c}

for key, (f, df, a, h) in cases.items():
    lin = f(a) + df(a) * h
    if check(f'P2.1({key})', yours[key], lin) and predicts[key] is not None:
        true = f(a + h)
        actual = 'over' if lin > true else 'under'
        verdict = '✓' if predicts[key] == actual else '✗'
        print(f"      true value {true:.10f}, so the approximation is {actual}.  "
              f"You predicted {predicts[key]} {verdict}.   E(h) = {true - lin:+.3e}")

How fast does $E(h)$ go to $0$ as $h\to 0$?

> **✏️ P2.2**
> (a) Using **only the definition of the derivative**, show that $\dfrac{E(h)}{h} \to 0$ as $h \to 0$, for any $f$ that is differentiable at $a$. (So $E(h)$ goes to $0$ *faster* than $h$.)
> (b) The next cell zooms in on $e^x$ and its tangent line at $a=0$, in windows $[0,h]$ for $h = 1,\ 0.1,\ 0.01$. The gap at the right edge of the window is $E(h)$. Suppose $E(h)\approx C\,h^p$. By what factor does the gap shrink each time the window shrinks by a factor of 10? Given (a), what is the smallest $p$ could be? Write down your prediction for the shrink factor.

In [ ]:
shrink_paper = None   # <- P2.2(b): predicted factor by which the gap shrinks per zoom step

E_exp = lambda h: np.exp(h) - 1 - h
if answered(shrink_paper):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, h in zip(axes, [1, 0.1, 0.01]):
        xs = np.linspace(0, h, 200)
        ax.plot(xs, np.exp(xs), label='e^x')
        ax.plot(xs, 1 + xs, '--', label='tangent 1 + x')
        ax.set_title(f'window [0, {h}]:  gap E(h) = {E_exp(h):.2e}')
        ax.legend()
    plt.tight_layout(); plt.show()

    gaps = [E_exp(h) for h in [1, 0.1, 0.01, 0.001]]
    for g1, g2 in zip(gaps, gaps[1:]):
        compare_prediction('P2.2(b) shrink factor', shrink_paper, g1 / g2)
else:
    _missing('P2.2(b)')

> **✏️ P2.3** For the three functions in P2.1, compute $f''(a)$ exactly: $f(x)=\sqrt x$ at $a=100$, $f(x)=e^x$ at $a=0$, $f(x)=\ln x$ at $a=1$.
>
> The next cell prints a table of $E(h)$, $E(h)/h$ and $E(h)/h^2$ for shrinking $h$. Look at the last column, and then do P2.4.

In [ ]:
f2_a_paper = None    # <- P2.3: f''(100) for sqrt(x)
f2_b_paper = None    # <- P2.3: f''(0)   for e^x
f2_c_paper = None    # <- P2.3: f''(1)   for ln(x)

f2_true = {'a': -0.25 * 100.0**(-1.5), 'b': 1.0, 'c': -1.0}
ok = [check('P2.3 sqrt', f2_a_paper, f2_true['a']),
      check('P2.3 exp',  f2_b_paper, f2_true['b']),
      check('P2.3 ln',   f2_c_paper, f2_true['c'])]

if all(ok):
    for key, (f, df, a, _) in cases.items():
        print(f"\n{['sqrt(x) at a = 100', 'e^x at a = 0', 'ln(x) at a = 1'][ord(key) - ord('a')]}")
        print(f"{'h':>10} {'E(h)':>14} {'E(h)/h':>14} {'E(h)/h^2':>14}")
        for h in [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]:
            E = f(a + h) - f(a) - df(a) * h
            print(f"{h:10.0e} {E:14.4e} {E / h:14.4e} {E / h**2:14.6f}")

> **✏️ P2.4** *(reflect)* In each table, $E(h)/h$ goes to $0$, as you proved in P2.2(a). The last column seems to settle down to a number. Compare those numbers with your values of $f''(a)$ from P2.3, and write down a conjecture of the form
> $$\lim_{h\to 0}\frac{E(h)}{h^2} = \ \text{(something involving } f''(a)\text{)}.$$
> (In the $\sqrt{x}$ table the last few rows drift. That is floating-point round-off: $E(h)$ there is a tiny difference of two numbers near $10$.)

## Part 3: An error bound from the MVT, applied twice

Your conjecture in P2.4 says $E(h)$ is roughly a constant times $h^2$. We now prove a rigorous version of that. The idea is to apply the Mean Value Theorem twice: once to $f$, and then once more to the derivative $f'$.

> **Theorem (error of linear approximation).** Suppose $f$ is twice differentiable on an interval $I$ containing $a$ and $a+h$, and $|f''(x)| \le M_2$ for all $x\in I$. Then
> $$|E(h)| = \bigl|f(a+h) - f(a) - f'(a)\,h\bigr| \;\le\; M_2\,h^2 .$$

You will prove this yourself in three steps.

> **✏️ P3.1 (Step 1: MVT for $f$).** Apply the MVT to $f$ on the interval between $a$ and $a+h$. Conclude that there is a point $\xi$ strictly between $a$ and $a+h$ with
> $$E(h) = \bigl(f'(\xi) - f'(a)\bigr)\,h .$$
>
> **✏️ P3.2 (Step 2: MVT for $f'$).** Explain why the MVT may be applied to the function $f'$ on the interval between $a$ and $\xi$. (Which hypotheses must you check, and why do they hold?) Conclude that there is a point $\eta$ between $a$ and $\xi$ with
> $$f'(\xi) - f'(a) = f''(\eta)\,(\xi - a).$$
>
> **✏️ P3.3 (Step 3: combine).** You now have $E(h) = f''(\eta)\,(\xi - a)\,h$.
> (a) Use $|\xi-a| \le |h|$ to prove the theorem.
> (b) Show that $(\xi - a)\,h > 0$, whatever the sign of $h$. Conclude: if $f''>0$ on $I$, then $E(h)>0$, so the tangent line lies **below** the graph. What if $f''<0$?
> (c) *(reflect)* Use (b) to justify your over/under predictions in P2.1, and to explain why $10.05$ *had* to be an upper bound for $\sqrt{101}$ in P1.5.

### Where did the factor $\tfrac12$ go?

Your conjecture in P2.4 has a $\tfrac12$ in it, but the theorem has constant $1$. We lost the $\tfrac12$ in P3.3(a) when we replaced $|\xi - a|$ by the cruder $|h|$. For $f(x)=e^x$ and $a=0$ we can find $\xi$ explicitly and see how much we gave away.

> **✏️ P3.4** Let $f(x)=e^x$, $a=0$.
> (a) From $f(h)-f(0) = f'(\xi)\,h$, solve for $\xi$ as a function of $h$.
> (b) Compute $\xi$ for $h=1$ as a decimal. You have seen this number before. Where?
> (c) Predict $\lim_{h\to 0} \xi/h$. (A guess is fine. Think about what $\xi/h$ means geometrically: where in the interval $[0,h]$ does $\xi$ sit?)

In [ ]:
xi1_paper    = None   # <- P3.4(b): xi for h = 1
xi_lim_guess = None   # <- P3.4(c): your prediction for lim xi/h

xi = lambda h: np.log(np.expm1(h) / h)     # expm1(h) = e^h - 1, computed without round-off
check('P3.4(b)', xi1_paper, xi(1.0))

if answered(xi_lim_guess):
    print(f"\n{'h':>10} {'xi(h)':>16} {'xi(h)/h':>12}")
    for h in [1, 1e-1, 1e-2, 1e-3, 1e-4]:
        print(f"{h:10.0e} {xi(h):16.6e} {xi(h) / h:12.6f}")
    compare_prediction('P3.4(c)', xi_lim_guess, xi(1e-4) / 1e-4)
else:
    _missing('P3.4(c)')

### Using the bound

> **✏️ P3.5** For each case (a), (b), (c) of P2.1: find a valid $M_2$ (the maximum of $|f''|$ on the interval between $a$ and $a+h$, or any honest upper bound for that maximum), and the resulting error bound $M_2 h^2$.
> For (b) you may not use the value of $e^{0.1}$ itself, since that is what you are approximating. Use a crude but honest bound instead, for example $e^{0.1} < e < 3$.

In [ ]:
bound_a_paper = None   # <- P3.5(a): your bound M2 * h^2 for sqrt(101)
bound_b_paper = None   # <- P3.5(b): your bound for e^0.1
bound_c_paper = None   # <- P3.5(c): your bound for ln(0.9)

bounds = {'a': bound_a_paper, 'b': bound_b_paper, 'c': bound_c_paper}
for key, (f, df, a, h) in cases.items():
    E = f(a + h) - f(a) - df(a) * h
    check_bound(f'P3.5({key})', bounds[key], E)

### An error-budget challenge (compare the $\varepsilon$–$N$ challenge from Week 3)

> **✏️ P3.6** You want to use $e^h \approx 1+h$ for **every** $h$ in $[0, h_{\max}]$, with guaranteed error less than $\varepsilon = 10^{-6}$. Use the theorem to find an $h_{\max}$ that is guaranteed to work. You may assume $h_{\max}\le 1$, so that $M_2 = e^{h_{\max}} \le e$ is a valid bound.
> *(reflect, after running)* The cell will tell you the largest $h_{\max}$ that actually works. Yours should be smaller by a factor of roughly $2.3$. Account for that factor. (Hint: there are two places where the bound gave something away, and $\sqrt{2e} \approx 2.33$.)

In [ ]:
hmax_paper = None    # <- P3.6: your guaranteed h_max

EPS = 1e-6
if answered(hmax_paper):
    # E(h) = e^h - 1 - h is increasing for h >= 0, so its maximum on [0, h_max] is E(h_max)
    worst = np.expm1(hmax_paper) - hmax_paper
    if worst < EPS:
        print(f"✓ Valid: for all 0 <= h <= {hmax_paper:.6g}, the error is at most {worst:.4e} < {EPS:g}.")
    else:
        print(f"✗ Not valid: at h = {hmax_paper:.6g} the error is {worst:.4e} >= {EPS:g}.")
    best = brentq(lambda h: np.expm1(h) - h - EPS, 1e-6, 0.1)
    print(f"  The largest h_max that actually works is about {best:.6g}"
          f"  (ratio to yours: {best / hmax_paper:.3g}).")
else:
    _missing('P3.6')

### Optional honors challenge: the sharp constant

> **✏️ P3.7 (challenge)** Prove the sharper bound $|E(h)| \le \tfrac12 M_2 h^2$ for $h>0$.
> *Hint.* By Step 2, $|f'(a+t)-f'(a)| \le M_2\,t$ for $0\le t\le h$. Now let
> $$\psi(t) = f(a+t) - f(a) - f'(a)\,t - \tfrac12 M_2\,t^2 .$$
> Compute $\psi(0)$ and $\psi'(t)$, show $\psi'(t)\le 0$, and use the MVT on $\psi$ (not on $f$!) to show $\psi(h)\le 0$. That gives $E(h)\le \frac12 M_2h^2$. A second function handles $E(h) \ge -\frac12 M_2 h^2$.
>
> Before running the next cell: what is the largest value you expect the plotted ratio to reach?

In [ ]:
ratio_guess = None   # <- P3.7: the largest value you expect the ratio to reach

# |E(h)| / (M2 h^2) for e^x at a = 0, with M2 = e^h, the exact max of |f''| on [0, h].
if answered(ratio_guess):
    hs = np.linspace(1e-4, 1, 500)
    ratio = (np.expm1(hs) - hs) / (np.exp(hs) * hs**2)
    plt.plot(hs, ratio, label='|E(h)| / (M2 h^2)')
    plt.axhline(1.0, color='k', lw=0.8, label='MVT-twice constant: 1')
    plt.axhline(0.5, color='r', ls='--', label='sharp constant: 1/2')
    plt.ylim(0, 1.1); plt.xlabel('h'); plt.legend()
    plt.title('How much of the error bound is actually used (e^x at a = 0)')
    plt.show()
    compare_prediction('P3.7', ratio_guess, ratio.max())
else:
    _missing('P3.7')

## Part 4: Euler's method, with an error bound

### The setup

An initial value problem asks for a function $y$ with
$$
y'(x) = F\bigl(x, y(x)\bigr), \qquad y(x_0) = y_0 .
$$
In Week 9, $F(x,y) = y$. Euler's method fixes a step size $h$, sets $x_k = x_0 + k h$, and computes
$$
y_{k+1} = y_k + h\,F(x_k, y_k).
$$
This is exactly the linear approximation from Part 2: $y(x_k + h) \approx y(x_k) + h\,y'(x_k) = y(x_k) + h\,F\bigl(x_k, y(x_k)\bigr)$, except that we do not know $y(x_k)$, so we use our approximation $y_k$ in its place. So there are **two** sources of error:

1. each step makes a linearization error, which we control with Part 3;
2. each step starts from the wrong place, $y_k$ instead of $y(x_k)$, so old errors are carried forward and can grow.

### Today's example

$$
y' = y\cos x, \qquad y(0) = 1 .
$$
This is harder than $y'=y$: the "growth rate" $\cos x$ changes sign, so the solution grows and decays in turn.

> **✏️ P4.1**
> (a) Verify that $y(x) = e^{\sin x}$ solves this initial value problem.
> (b) Find the maximum and minimum values of $y$ on $\mathbb{R}$ and where they occur.
> (c) Sketch $y$ on $[0, 2\pi]$. Mark the points where $y' = 0$.

In [ ]:
ymax_paper = None   # <- P4.1(b): maximum value of y
ymin_paper = None   # <- P4.1(b): minimum value of y

F = lambda x, y: y * np.cos(x)
y_exact = lambda x: np.exp(np.sin(x))

xs_fine = np.linspace(0, 2 * np.pi, 2001)
ok = [check('P4.1(b) max', ymax_paper, y_exact(xs_fine).max()),
      check('P4.1(b) min', ymin_paper, y_exact(xs_fine).min())]

if all(ok):
    # direction field: a short segment of slope F(x, y) through each grid point
    X, Y = np.meshgrid(np.linspace(0, 2 * np.pi, 25), np.linspace(0.2, 3.0, 15))
    S = F(X, Y)
    L = np.sqrt(1 + S**2)
    plt.quiver(X, Y, 1 / L, S / L, angles='xy', color='gray', alpha=0.6, headwidth=1, headlength=0)
    plt.plot(xs_fine, y_exact(xs_fine), lw=2, label='y = exp(sin x)')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend()
    plt.title("Direction field of y' = y cos x and the exact solution")
    plt.show()

> **✏️ P4.2 Euler by hand.** Use $h = \pi/4$ and four steps, so that $x_4 = \pi$.
> (a) Show that each step is $y_{k+1} = y_k\,(1 + h\cos x_k)$.
> (b) Compute $y_1, y_2, y_3, y_4$ **exactly** (in terms of $\pi$ and $\sqrt2$), then as decimals.
> (c) One of the steps leaves $y$ unchanged. Which one, and why? How can you see this in the direction field?
> (d) Compare $y_4$ with $y(\pi)$. Is Euler's approximation too high or too low?

In [ ]:
def euler(F, x0, y0, h, n):
    # n steps of Euler's method with step size h. Returns arrays x_0..x_n and y_0..y_n.
    xs = x0 + h * np.arange(n + 1)
    ys = np.empty(n + 1)
    ys[0] = y0
    for k in range(n):
        ys[k + 1] = ys[k] + h * F(xs[k], ys[k])
    return xs, ys


y_paper = None    # <- P4.2(b): the list [y1, y2, y3, y4], in order

h = np.pi / 4
xs, ys = euler(F, 0.0, 1.0, h, 4)
if check_list('P4.2(b)', y_paper, ys[1:], ordered=True):
    t = np.linspace(0, np.pi, 300)
    plt.plot(t, y_exact(t), label='exact  exp(sin x)')
    plt.plot(xs, ys, 'o-', label='Euler, h = pi/4')
    plt.xlabel('x'); plt.legend(); plt.title('Four Euler steps by hand')
    plt.show()
    print(f"error at x = pi:  y(pi) - y_4 = {y_exact(np.pi) - ys[-1]:+.6f}")

> **✏️ P4.3** *(predict)* Based on what we saw in Week 9, predict the error $y(\pi) - y_n$ at $x=\pi$ when $h = \pi/40$, ten times smaller than in P4.2.

In [ ]:
err40_guess = None   # <- P4.3: your predicted error at x = pi for h = pi/40

def euler_error_at(X, n):
    xs, ys = euler(F, 0.0, 1.0, X / n, n)
    return y_exact(X) - ys[-1]

compare_prediction('P4.3', err40_guess, euler_error_at(np.pi, 40))

if answered(err40_guess):
    ns = 4 * 2 ** np.arange(0, 9)
    hs = np.pi / ns
    errs = np.array([abs(euler_error_at(np.pi, n)) for n in ns])
    slope, _ = np.polyfit(np.log(hs), np.log(errs), 1)
    plt.loglog(hs, errs, 'o-', label='|error| at x = pi')
    plt.loglog(hs, hs * errs[0] / hs[0], '--', label='reference line, slope 1')
    plt.xlabel('step size h'); plt.ylabel('|error|'); plt.legend()
    plt.title(f"Euler for y' = y cos x: fitted slope {slope:.3f}")
    plt.show()

### Why is the error proportional to $h$?

Fix $h$ and write $y(x)$ for the exact solution. Define

- the **global error** after $k$ steps: $e_k = y(x_k) - y_k$, with $e_0 = 0$;
- the **local error** of step $k$: $\tau_k = y(x_{k+1}) - y(x_k) - h\,y'(x_k)$.

The local error $\tau_k$ is just the linearization error $E(h)$ from Part 2, for the function $y$ at the point $a=x_k$. It measures what a single Euler step would get wrong *if it started from the exact value*.

> **✏️ P4.4**
> (a) Use the theorem from Part 3 to show $|\tau_k| \le M_2 h^2$, where $M_2$ is any bound for $|y''|$ on the whole interval we are working on.
> (b) For $y = e^{\sin x}$, show that $y'' = e^{\sin x}\,(\cos^2 x - \sin x)$.
> (c) Find $M_2 = \max |y''|$ on $[0, \pi]$. *Hint:* substitute $s = \sin x \in [0,1]$ and $\cos^2 x = 1 - s^2$, and find the extreme values of $g(s) = e^s(1-s-s^2)$ on $[0,1]$. If you get stuck, a cruder valid bound (for example $M_2 \le 2e$) is acceptable, as long as you justify it.
> (d) Explain why $e_1 = \tau_0$, and compute $e_1$ for $h = \pi/4$ using your answer to P4.2.

In [ ]:
M2_paper = None    # <- P4.4(c): your bound for |y''| on [0, pi]
e1_paper = None    # <- P4.4(d): e_1 for h = pi/4

ypp = lambda x: np.exp(np.sin(x)) * (np.cos(x)**2 - np.sin(x))
t = np.linspace(0, np.pi, 100001)
check_bound('P4.4(c)', M2_paper, np.abs(ypp(t)).max())

h = np.pi / 4
xs, ys = euler(F, 0.0, 1.0, h, 4)
check('P4.4(d)', e1_paper, y_exact(xs[1]) - ys[1])

if answered(M2_paper):
    tau = y_exact(xs[1:]) - y_exact(xs[:-1]) - h * F(xs[:-1], y_exact(xs[:-1]))
    print(f"\nlocal errors tau_k for h = pi/4: {np.round(tau, 5)}")
    check_bound('P4.4(a) for every k', M2_paper * h**2, np.abs(tau).max())

> **✏️ P4.5** Derive the **exact** error recursion
> $$e_{k+1} = (1 + h\cos x_k)\,e_k + \tau_k .$$
> *Hint:* write $y(x_{k+1}) = y(x_k) + h\,y'(x_k) + \tau_k$ (that is the definition of $\tau_k$), use $y'(x_k) = y(x_k)\cos x_k$, and subtract the Euler step $y_{k+1} = y_k + h\,y_k\cos x_k$.
> Then conclude that $|e_{k+1}| \le (1+h)\,|e_k| + M_2 h^2$.
>
> The next cell checks your recursion numerically for $h = \pi/40$: if it is right, the "leftover" $e_{k+1} - (1+h\cos x_k)e_k - \tau_k$ should be zero up to round-off (around $10^{-15}$). Enter the multiplier $1 + h\cos x_k$ as you derived it, as a Python function of `h` and `x`.

In [ ]:
multiplier_paper = None   # <- P4.5: e.g.  lambda h, x: 1 + h * np.cos(x)

if answered(multiplier_paper):
    h, n = np.pi / 40, 40
    xs, ys = euler(F, 0.0, 1.0, h, n)
    e   = y_exact(xs) - ys
    tau = y_exact(xs[1:]) - y_exact(xs[:-1]) - h * F(xs[:-1], y_exact(xs[:-1]))
    leftover = e[1:] - multiplier_paper(h, xs[:-1]) * e[:-1] - tau
    print(f"largest |leftover| over all {n} steps: {np.abs(leftover).max():.2e}")
    print("✓ recursion confirmed" if np.abs(leftover).max() < 1e-12 else "✗ recursion does not hold: recheck P4.5")
else:
    _missing('P4.5')

> **✏️ P4.6** From here on the argument is pure bookkeeping.
> (a) Starting from $e_0 = 0$, prove by induction on $n$ that
> $$|e_n| \;\le\; M_2 h^2\bigl(1 + (1+h) + (1+h)^2 + \dots + (1+h)^{n-1}\bigr) \;=\; M_2\,h\,\bigl((1+h)^n - 1\bigr).$$
> (b) Use the MVT for $e^x$ on $[0,h]$ to show $1 + h \le e^h$ for $h\ge 0$. Conclude $(1+h)^n \le e^{nh}$.
> (c) Conclude that, since $x_n = nh$,
> $$|e_n| \;\le\; M_2\,\bigl(e^{x_n} - 1\bigr)\,h .$$
> (d) *(reflect)* At a fixed point $x_n = X$, the bound in (c) is (constant) $\times\, h$. Explain in two sentences why this accounts for the slope-1 line in P4.3 and in Week 9.
>
> **✏️ P4.7** Evaluate the bound (c) at $X = \pi$ for $h = \pi/4$ and for $h = \pi/40$, using your $M_2$. By roughly what factor does each overestimate the actual error?

In [ ]:
bound4_paper  = None   # <- P4.7: the bound at x = pi for h = pi/4
bound40_paper = None   # <- P4.7: the bound at x = pi for h = pi/40

check_bound('P4.7, h = pi/4',  bound4_paper,  euler_error_at(np.pi, 4))
check_bound('P4.7, h = pi/40', bound40_paper, euler_error_at(np.pi, 40))

The bound is valid but very pessimistic. Let us see where it gives so much away.

In P4.5, the honest inequality is $|e_{k+1}| \le |1 + h\cos x_k|\,|e_k| + M_2h^2$. In P4.6 we replaced the multiplier $|1+h\cos x_k|$ by $1+h$, the worst it can ever be. If instead we keep the true multiplier, we get a **recursive bound** $b_k$ that the computer can evaluate step by step:
$$
b_0 = 0, \qquad b_{k+1} = |1 + h\cos x_k|\, b_k + M_2 h^2 .
$$

> **✏️ P4.8**
> (a) Prove by induction that $|e_k| \le b_k$ for all $k$.
> (b) For which $x_k$ is the multiplier $|1+h\cos x_k|$ less than $1$ (take $h<1$)? What does such a step do to an error that is already there?
> (c) *(predict)* The next cell plots $|e_k|$, $b_k$ and the bound from P4.6(c) on the longer interval $[0, 4\pi]$, for $h = \pi/40$. Before running it, sketch on paper what you expect the three curves to look like, on a logarithmic vertical axis. In particular: does the actual error keep growing, or does it come back down? (Look at your sketch from P4.1(c).)

In [ ]:
if answered(M2_paper):
    h, n = np.pi / 40, 160                      # x from 0 to 4 pi
    xs, ys = euler(F, 0.0, 1.0, h, n)
    e = np.abs(y_exact(xs) - ys)

    b = np.zeros(n + 1)                          # recursive bound
    for k in range(n):
        b[k + 1] = abs(1 + h * np.cos(xs[k])) * b[k] + M2_paper * h**2
    crude = M2_paper * h * (np.exp(xs) - 1)      # bound from P4.6(c); M2 = e works on all of [0, 4 pi]

    plt.semilogy(xs[1:], e[1:], label='actual |e_k|')
    plt.semilogy(xs[1:], b[1:], label='recursive bound b_k')
    plt.semilogy(xs[1:], crude[1:], label='crude bound M2 h (e^x - 1)')
    for j in range(1, 5):
        plt.axvline(j * np.pi, color='gray', lw=0.5)
    plt.xlabel('x  (gray lines at multiples of pi)'); plt.legend()
    plt.title('Euler error for y\' = y cos x, h = pi/40')
    plt.show()
    print(f"at x = 4 pi:  actual {e[-1]:.3e},  recursive bound {b[-1]:.3e},  crude bound {crude[-1]:.3e}")
    print("(M2 = e bounds |y''| on all of R, so it is valid on [0, 4 pi] too.)")
else:
    _missing('P4.4(c)')

> **✏️ P4.9** *(reflect)*
> (a) Both bounds are rigorous. Both are proportional to $h$. Why might we still prefer the crude one in a proof, and the recursive one in practice?
> (b) The crude bound grows like $e^{x}$ because it pretends the solution is always growing as fast as $y'=y$ allows. Point to the exact step in P4.5–P4.6 where that pretending happens.

## Extension A: Euler's method is a Riemann sum in disguise

> **✏️ PA.1**
> (a) Show that for $y'=y\cos x$, $y(0)=1$, Euler's method gives the product $y_n = \prod_{k=0}^{n-1}(1 + h\cos x_k)$, and therefore (for $h<1$, so every factor is positive)
> $$\ln y_n = \sum_{k=0}^{n-1} \ln\bigl(1 + h\cos x_k\bigr).$$
> (b) Use the linear approximation $\ln(1+u) \approx u$ (this is P2.1(c) in disguise) to get $\ln y_n \approx h\sum_{k=0}^{n-1}\cos x_k$. This is a left Riemann sum. Of which integral? What is the value of that integral? Conclude $y_n \approx e^{\sin x_n}$.
> (c) *(challenge)* Use Part 3 on $\ln(1+u)$ with $|u|\le h$ to show that each term's linearization error is at most $\frac{h^2}{(1-h)^2}$, so the total over $n$ steps is at most $\frac{x_n\,h}{(1-h)^2}$. This is another first-order error, found in a completely different way.
>
> (d) *(predict)* The next cell plots $\ln y_n$, the Riemann sum $h\sum\cos x_k$, and $\sin x_n$ on $[0, 4\pi]$ for $h=\pi/8$. Before running it, predict which two of the three curves will be closest to each other. *(reflect, after running)* Were you right? Use (c) to explain the steady downward drift of $\ln y_n$. (Second-order hint: $\ln(1+u) \approx u - \tfrac12 u^2$.)

In [ ]:
closest_guess = None   # <- PA.1: which two curves will be closest? e.g. 'ln y_n and sin x'

if answered(closest_guess):
    h, n = np.pi / 8, 32
    xs, ys = euler(F, 0.0, 1.0, h, n)
    riemann = np.concatenate([[0.0], h * np.cumsum(np.cos(xs[:-1]))])

    plt.plot(xs, np.log(ys), 'o-', label='ln y_n (Euler)')
    plt.plot(xs, riemann, 's--', label='left Riemann sum of cos')
    t = np.linspace(0, 4 * np.pi, 400)
    plt.plot(t, np.sin(t), 'k', lw=1, label='sin x')
    plt.xlabel('x'); plt.legend(); plt.title('Euler for y\' = y cos x, viewed through ln')
    plt.show()
    print(f"your guess: {closest_guess}")
    print(f"max |ln y_n - Riemann sum|  = {np.abs(np.log(ys) - riemann).max():.4f}")
    print(f"max |Riemann sum - sin x_n| = {np.abs(riemann - np.sin(xs)).max():.4f}")
    print(f"max |ln y_n - sin x_n|      = {np.abs(np.log(ys) - np.sin(xs)).max():.4f}")
    print(f"PA.1(c) bound for |ln y_n - Riemann sum| at x = 4 pi: {4 * np.pi * h / (1 - h)**2:.4f}")
else:
    _missing('PA.1')

## Extension B: falling with air resistance

A skydiver falls from rest. With quadratic air drag, the downward velocity $v(t)$ (in m/s) satisfies
$$
v' = g - k\,v^2, \qquad v(0) = 0,
$$
with $g = 9.8\ \text{m/s}^2$ and, for a typical skydiver, $k = 0.0039\ \text{m}^{-1}$.

> **✏️ PB.1**
> (a) Find the **terminal velocity** $v_T > 0$, the constant solution of $v' = g - kv^2$. Give it exactly and as a decimal.
> (b) Explain, from the equation alone, why a solution that starts at $v(0)=0$ is increasing and can never exceed $v_T$.
> (c) Let $\tanh u = \dfrac{e^u - e^{-u}}{e^u + e^{-u}}$. Show that $\dfrac{d}{du}\tanh u = 1 - \tanh^2 u$, and use it to verify that $v(t) = v_T \tanh\!\left(\dfrac{g\,t}{v_T}\right)$ solves the problem. (Use $k\,v_T^2 = g$.)

In [ ]:
vT_paper = None    # <- PB.1(a): terminal velocity in m/s

g, k = 9.8, 0.0039
vT = np.sqrt(g / k)
G = lambda t, v: g - k * v**2
v_exact = lambda t: vT * np.tanh(g * t / vT)

if check('PB.1(a)', vT_paper, vT):
    T = 30
    t = np.linspace(0, T, 400)
    plt.plot(t, v_exact(t), 'k', lw=2, label='exact')
    for h in [3.0, 1.0]:
        ts, vs = euler(G, 0.0, 0.0, h, int(round(T / h)))
        plt.plot(ts, vs, 'o-', ms=3, label=f'Euler, h = {h}')
    plt.axhline(vT, color='gray', ls=':', label='terminal velocity')
    plt.xlabel('t (s)'); plt.ylabel('v (m/s)'); plt.legend()
    plt.title('Skydiver with quadratic drag')
    plt.show()

For this problem the crude bound of Part 4 is hopeless (try it: with Lipschitz constant $2kv_T$ it grows like $e^{2kv_T t}$, and $e^{2kv_T\cdot 30}\approx 10^{5}$). But this equation has a feature we can exploit: drag *pulls errors back*.

> **✏️ PB.2**
> (a) Write $e_k = v(t_k) - v_k$. Show that
> $$e_{k+1} = \bigl(1 - h\,k\,(v(t_k) + v_k)\bigr)\,e_k + \tau_k .$$
> (Hint: factor a difference of squares. This is the MVT for $G(v) = g-kv^2$ in disguise: $G(v)-G(w) = G'(\zeta)(v-w)$, and here $\zeta$ is exactly the midpoint of $v$ and $w$.)
> (b) *(challenge)* Show that if $0\le v_k < v_T$ and $h \le \dfrac{1}{2kv_T}$, then $v_k \le v_{k+1} < v_T$. Hint: $v_T - v_{k+1} = (v_T - v_k)\bigl(1 - hk(v_T + v_k)\bigr)$. So Euler's values stay in $[0, v_T)$, just like the true solution. Look back at the $h = 3$ curve in the previous plot: what happens there, and which condition fails?
> (c) Conclude that for $h \le \frac{1}{2kv_T}$ the multiplier in (a) lies in $[0,1]$, so $|e_{k+1}| \le |e_k| + M_2h^2$, and therefore $|e_n| \le M_2\,h\,t_n$. Compare this with P4.6(c): where did the exponential go?
>
> **✏️ PB.3** Compute $M_2 = \max|v''|$. *Hint:* differentiate the equation to get $v'' = -2kv\,v' = -2kv(g - kv^2)$, and maximize over $0 \le v \le v_T$. Show the maximum is $M_2 = \dfrac{4g^2}{3\sqrt3\,v_T}$ and evaluate it.
>
> **✏️ PB.4 (design problem).** Choose a step size $h$ for which PB.2(c) **guarantees** that the Euler velocity is within $0.1$ m/s of the true velocity for all $0 \le t \le 30$ s. Check that your $h$ satisfies the condition $h \le \frac{1}{2kv_T}$ as well.

In [ ]:
M2v_paper = None   # <- PB.3: max |v''|
h_paper   = None   # <- PB.4: your guaranteed step size

M2v = 4 * g**2 / (3 * np.sqrt(3) * vT)
check('PB.3', M2v_paper, M2v)

T, TOL = 30.0, 0.1
if answered(h_paper) and h_paper < 1e-4:
    print(f"h = {h_paper:g} would need {T / h_paper:.0f} steps, far more than necessary. Recheck PB.4.")
elif answered(h_paper):
    print(f"condition h <= 1/(2 k vT) = {1 / (2 * k * vT):.4f}:  {'✓ holds' if h_paper <= 1 / (2 * k * vT) else '✗ fails'}")
    n = int(np.ceil(T / h_paper))
    ts, vs = euler(G, 0.0, 0.0, h_paper, n)
    worst = np.abs(v_exact(ts) - vs).max()
    print(f"Euler values stay in [0, vT): {'✓' if (vs.min() >= 0 and vs.max() < vT) else '✗'}")
    print(f"max error on [0, {T:g}] with h = {h_paper:.5g}: {worst:.4e} m/s  "
          f"{'✓ within 0.1 m/s' if worst <= TOL else '✗ NOT within 0.1 m/s'}")

    # How large could h actually have been? (largest h on a grid that still meets the tolerance)
    def max_err(h):
        ts, vs = euler(G, 0.0, 0.0, h, int(np.ceil(T / h)))
        return np.abs(v_exact(ts) - vs).max()
    trial = np.geomspace(h_paper, 2.0, 200)
    works = [h for h in trial if max_err(h) <= TOL]
    print(f"the largest h (on a trial grid) that actually meets the tolerance is about {max(works):.3g} "
          f"(ratio to yours: {max(works) / h_paper:.3g})")
else:
    _missing('PB.4')

> **✏️ PB.5** *(reflect)* Your guaranteed $h$ is smaller than the largest one that actually works, by the factor printed above. List the places in the argument where we gave something away. Which do you think costs the most? (Compare P3.6 and P4.9.)

## Final reflection

> **✏️ F.1** Write a short paragraph (on paper) tracing the chain of ideas in this lab: from the MVT, to the error bound for linear approximation, to the error bound for Euler's method. At each link, say in one sentence which theorem or estimate carries the argument.
>
> **✏️ F.2** The MVT only says that a point $c$ *exists*; it never tells us where. Yet every error bound in this lab was a concrete number we could check by computer. Explain how an existence statement can produce a concrete numerical guarantee.